<a href="https://colab.research.google.com/github/Harshavardhan-gowdu-v/Harsha-POC-CODE/blob/main/chat_with_PDF_by_Gardio_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pymupdf faiss-cpu pillow sentence-transformers transformers gradio

In [2]:
import fitz
import os
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
import faiss
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [ ]:
file_path = "/content/sample_data/ARCS Data sample.pdf"

doc = fitz.open(file_path)

texts = []
image_paths = []

img_folder = "/content/pdf_images"
os.makedirs(img_folder, exist_ok=True)

for page_num in range(len(doc)):
    page = doc[page_num]
    texts.append(page.get_text())

    for i, img in enumerate(page.get_images(full=True)):
        xref = img[0]
        base = doc.extract_image(xref)
        img_bytes = base["image"]
        img_ext = base.get("ext", "png")

        path = f"{img_folder}/p{page_num}_{i}.{img_ext}"
        with open(path, "wb") as f:
            f.write(img_bytes)

        image_paths.append((page_num, path))

print(f"✅ Pages: {len(texts)}")
print(f"✅ Images: {len(image_paths)}")
print(f"✅ Sample text from page 0: {texts[0][:200]}")

In [ ]:
def split_text(text, chunk_size=500, chunk_overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - chunk_overlap
    return chunks

chunks = []
chunk_page_map = []

for i, text in enumerate(texts):
    split_chunks = split_text(text)
    for c in split_chunks:
        chunks.append(c)
        chunk_page_map.append(i)

print(f"✅ Total chunks: {len(chunks)}")
print(f"✅ Sample chunk: {chunks[0][:200]}")

In [ ]:
text_model = SentenceTransformer("all-mpnet-base-v2")
text_embeddings = text_model.encode(chunks, normalize_embeddings=True)

dim = text_embeddings.shape[1]
text_index = faiss.IndexFlatIP(dim)
text_index.add(np.array(text_embeddings))

print(f"✅ Text index: {text_index.ntotal} vectors")

In [ ]:
clip_model = SentenceTransformer("clip-ViT-B-32")

image_embeddings = []
image_files = []

for page, path in image_paths:
    try:
        img = Image.open(path).convert("RGB")
        emb = clip_model.encode(img, normalize_embeddings=True)
        image_embeddings.append(emb)
        image_files.append((page, path))
    except:
        continue

image_embeddings = np.array(image_embeddings)

image_index = faiss.IndexFlatIP(image_embeddings.shape[1])
image_index.add(image_embeddings)

In [8]:
from transformers import pipeline

In [ ]:
qa_model = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)

In [10]:
chat_history = []

In [11]:
def chatbot(query):
    global chat_history

    # 🔹 TEXT SEARCH
    q_emb = text_model.encode([query], normalize_embeddings=True)
    D, I = text_index.search(np.array(q_emb), k=3)

    retrieved_chunks = [chunks[i] for i in I[0]]
    retrieved_pages = [chunk_page_map[i] for i in I[0]]

    context = "\n".join(retrieved_chunks)

    # 🔹 PROMPT
    prompt = f"""
You are an intelligent assistant.

Rules:
- Use ONLY the context
- If not found say: Not available in document
- Answer clearly

Context:
{context}

Question:
{query}
"""

    result = qa_model(prompt)[0]['generated_text']

    # 🔹 IMAGE FILTER (IMPORTANT FIX)
    q_img = clip_model.encode([query], normalize_embeddings=True)
    D_img, I_img = image_index.search(np.array(q_img), k=5)

    relevant_images = []

    for score, idx in zip(D_img[0], I_img[0]):
        if score > 0.25:  # 🔥 threshold fix
            relevant_images.append(image_files[idx])

    # 🔹 FILTER BY PAGE (IMPORTANT)
    final_images = [
        path for page, path in relevant_images if page in retrieved_pages
    ]

    # 🔹 SAVE CHAT HISTORY
    chat_history.append((query, result))
    chat_history = chat_history[-3:]

    return result, final_images, retrieved_pages

In [ ]:
import gradio as gr

def chat_ui(query):
    answer, images, pages = chatbot(query)

    return answer, images, str(pages)

interface = gr.Interface(
    fn=chat_ui,
    inputs="text",
    outputs=["text", "gallery", "text"],
    title="📄 Multimodal PDF Chatbot"
)

interface.launch()